# Hi-EF Phase 1 intervention statistical audit

Analyze saved validation logits from `ptrnghieu/hief-interventions`. This notebook uses CPU only, performs no training, and never evaluates the test partition.

In [ ]:
from pathlib import Path
import subprocess

REPO = Path('/kaggle/working/hi-ef-materials')
OUTPUT = Path('/kaggle/working/phase1_intervention_statistics')
if (REPO / '.git').exists():
    subprocess.run(['git', '-C', str(REPO), 'fetch', 'origin', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'checkout', 'experiments'], check=True)
    subprocess.run(['git', '-C', str(REPO), 'pull', '--ff-only', 'origin', 'experiments'], check=True)
else:
    subprocess.run([
        'git', 'clone', '--branch', 'experiments', '--single-branch',
        'https://github.com/ptrnghieu/hi-ef-materials.git', str(REPO)
    ], check=True)
print('Repository ready')

In [ ]:
candidates = list(Path('/kaggle/input').rglob('intervention_summary.json'))
valid = []
for summary_path in candidates:
    root = summary_path.parent
    if (root / 'predictions/full_seed42_true.npz').exists():
        valid.append(root)
if len(valid) != 1:
    raise RuntimeError(
        'Attach exactly one output from ptrnghieu/hief-interventions; '
        f'found {len(valid)} valid roots: {valid}'
    )
INTERVENTIONS = valid[0]
print('Intervention output:', INTERVENTIONS)

In [ ]:
command = [
    'python', str(REPO / 'experiments/analyze_validation_interventions.py'),
    '--interventions-dir', str(INTERVENTIONS),
    '--manifest', str(REPO / 'experiments/manifests/source_folder_split_seed42.csv'),
    '--output-dir', str(OUTPUT),
    '--bootstrap-replicates', '5000',
    '--bootstrap-seed', '2901'
]
subprocess.run(command, check=True)

In [ ]:
import json
import pandas as pd
summary = json.loads((OUTPUT / 'statistical_summary.json').read_text())
assert summary['test_evaluated'] is False
display(pd.read_csv(OUTPUT / 'condition_effects_by_model_seed.csv'))
display(pd.read_csv(OUTPUT / 'per_class_effects.csv').head(35))
print(json.dumps({
    'aggregate': summary['aggregate'],
    'hierarchical_bootstrap': summary['hierarchical_bootstrap'],
    'inference_note': summary['inference_note'],
}, indent=2))

Run via **Save Version → Save & Run All** with the `hief-interventions` output attached. A GPU and the feature dataset are not required. Preserve `/kaggle/working/phase1_intervention_statistics`.